In [21]:
from pathlib import Path
import pickle
from utility import make_model_cost_batch, make_models_cost_batch, _turnup_data, setpoint_df
from typing import Any, Dict, List, Optional
import pandas as pd

MODELS_DIR = Path("models").resolve()

with open(MODELS_DIR / "model_electricity.pkl", "rb") as f:
    electricity_model = pickle.load(f)
 
with open(MODELS_DIR / "model_steam.pkl", "rb") as f:
    steam_model = pickle.load(f)
 
with open(MODELS_DIR / "model_starch_top.pkl", "rb") as f:
    model_starch_top = pickle.load(f)

with open(MODELS_DIR / "model_starch_bottom.pkl", "rb") as f:
    model_starch_bottom = pickle.load(f)

with open(MODELS_DIR / "model_SCTCD.pkl", "rb") as f:
    model_SCTCD = pickle.load(f)

with open(MODELS_DIR / "model_cost_starch_bottom.pkl", "rb") as f:
    model_cost_starch_bottom = pickle.load(f)

with open(MODELS_DIR / "model_cost_starch_top.pkl", "rb") as f:
    model_cost_starch_top = pickle.load(f) 
 
# -------------------------------------------------
# Feature definitions
# -------------------------------------------------
 
def steam_features() -> List[str]:
    return list(steam_model.best_estimator_.feature_names_in_)
 
def electricity_features() -> List[str]:
    return list(electricity_model.best_estimator_.feature_names_in_)
 
def starch_features():
    return list(set(list(model_starch_top.best_estimator_.feature_names_in_) + list(model_starch_bottom.best_estimator_.feature_names_in_)))

def starch_cost_features():
    return list(set(list(model_cost_starch_bottom.best_estimator_.feature_names_in_) + list(model_cost_starch_top.best_estimator_.feature_names_in_)))
 
def fibre_features() -> List[str]:
    return [
        "Current_basis_weight",
        "Starch_uptake_by_paper_Bottom_Roll__g/m2_",
        "Starch_uptake_by_paper_Top_Roll__g/m2_",
        "Current_reel_moisture_average(reel)",
    ]

def SCTCD_features():
    return list(model_SCTCD.best_estimator_.feature_names_in_)
 
 
# -------------------------------------------------
# Formula-based component
# -------------------------------------------------
 
def fibre_cost_from_row(row) -> float:    
    #require_columns(row, fibre_features(), "fibre")    
    if type(row) == pd.Series:
        row = row.to_dict()
    basis_weight = row["Current_basis_weight"]
    starch_uptake = row["Starch_uptake_by_paper_Bottom_Roll__g/m2_"] + row["Starch_uptake_by_paper_Top_Roll__g/m2_"]
    moisture = row["Current_reel_moisture_average(reel)"]

    return 146.46 * (basis_weight * (1 - moisture / 100) - starch_uptake) / basis_weight


def fibre_cost(X):
    """
    Behaves like the model-based cost functions:
    - accepts one row: pd.Series / dict
    - accepts a dataframe: pd.DataFrame
    """    
    if isinstance(X, list):
        X = pd.DataFrame(X)
    if isinstance(X, pd.DataFrame):
        
        missing = [c for c in fibre_features() if c not in X.columns]
        if missing:
            raise KeyError(f"Missing columns for 'fibre': {missing}")

        basis_weight = X["Current_basis_weight"]
        starch_uptake = X["Starch_uptake_by_paper_Bottom_Roll__g/m2_"] + X["Starch_uptake_by_paper_Top_Roll__g/m2_"]
        moisture = X["Current_reel_moisture_average(reel)"]

        return 146.46 * (basis_weight * (1 - moisture / 100) - starch_uptake) / basis_weight    
    return float(fibre_cost_from_row(X))
 
 
# -------------------------------------------------
# Model-based components
# -------------------------------------------------
 
_, steam_cost = make_model_cost_batch(steam_model, steam_features)
_, electricity_cost = make_model_cost_batch(electricity_model, electricity_features)
_, starch_cost = make_models_cost_batch(
    models={
        "bottom": model_starch_bottom,
        "top": model_starch_top,
    },
    feature_fns={
        "bottom": list(model_starch_bottom.best_estimator_.feature_names_in_),
        "top": list(model_starch_top.best_estimator_.feature_names_in_),
    },
    agg="sum",
) 

_, sctcd_strength = make_model_cost_batch(model_SCTCD, SCTCD_features)




In [10]:
turnup_data = _turnup_data(True, "", None, setpoint_df, False)
turnup_data["MBS_SCT_MD_L1"] = turnup_data.MBS_SCT_MD.shift(1)
turnup_data["MBS_SCT_CD_L1"] = turnup_data.MBS_SCT_CD.shift(1)
turnup_data["MBS_Burst_L1"] = turnup_data.MBS_Burst.shift(1)
turnup_data["MBS_CMT30_L1"] = turnup_data.MBS_CMT30.shift(1)

In [17]:
sctcd_strength(turnup_data[SCTCD_features()].dropna())

array([2.00656953, 2.00436091, 2.01374642, ..., 2.0747224 , 2.08183625,
       2.06725698], shape=(4603,))

In [18]:
starch_cost(turnup_data[starch_features()].dropna())

array([4.92095515, 4.96119738, 4.97488824, ..., 6.03900549, 6.09669957,
       6.20853833], shape=(4604,))

In [24]:
fibre_cost(turnup_data[fibre_features()].dropna())

0       129.661476
1       126.930113
2       127.269643
3       129.859378
4       126.939827
           ...    
4599    127.568557
4600    127.627797
4601    127.684452
4602    127.069476
4603    128.157765
Length: 4604, dtype: float64